## GPT Scores

In [3]:
import pandas as pd
import numpy as np

df1 = pd.read_csv('SwissSurvey_Analysis_GPT_Measurements_4omini.csv')
df1 = df1.rename(columns={'Cost': 'Cost1'})
df1 = df1.rename(columns={'Date_log_gptapi': 'Date_log_gptapi1'})

df2 = pd.read_csv('SwissSurvey_Analysis_typo_4omini.csv')
df2 = df2.rename(columns={'Cost': 'Cost2'})
df2 = df2.rename(columns={'Date_log_gptapi': 'Date_log_gptapi2'})

def validate_ids(df1, df2, id_col='ResponseId'):
    if not df1[id_col].is_unique or not df2[id_col].is_unique:
        raise ValueError(f"Duplicate {id_col} found in one of the dataframes!")
    ids1 = set(df1[id_col])
    ids2 = set(df2[id_col])
    
    if ids1 != ids2:
        missing_in_df2 = ids1 - ids2
        missing_in_df1 = ids2 - ids1
        raise ValueError(
            f"ID mismatch! \n"
            f"IDs in df1 but not df2: {list(missing_in_df2)[:5]}... \n"
            f"IDs in df2 but not df1: {list(missing_in_df1)[:5]}..."
        )
    
    print("✅ Validation passed: ResponseIds match perfectly.")

validate_ids(df1, df2)

✅ Validation passed: ResponseIds match perfectly.


In [4]:
cols_to_add = [
    'ResponseId', 'Date_log_gptapi2', 'Cost2', 'Post-typos', 
    'Post-typos-explanation', 'Post-ai_likelihood', 'Post-ai_likelihood-explanation', 
    'ClimateLong-typos', 'ClimateLong-typos-explanation', 
    'ClimateLong-ai_likelihood', 'ClimateLong-ai_likelihood-explanation'
]

combined_df = pd.merge(df1, df2[cols_to_add], on='ResponseId', how='left')
combined_df['Cost'] = combined_df['Cost1'] + combined_df['Cost2']

starting_cols = [
    'ResponseId', 'ClimateLong', 'Post', 'model_name', 
    'Date_log_gptapi1', 'Date_log_gptapi2', 'Cost'
]

remaining_cols = [c for c in combined_df.columns if c not in starting_cols and c not in ['Cost1', 'Cost2']]
combined_df = combined_df[starting_cols + remaining_cols]
combined_df.columns = [col.replace('-', '_') for col in combined_df.columns]
combined_df.columns = [col.replace('ClimateLong_', 'Base_') if col.startswith('ClimateLong_') else col for col in combined_df.columns]
combined_df.columns = combined_df.columns.str.replace('_typos', '_typos_count')

In [5]:
combined_df.to_csv('SwissSurvey_Analysis_GPT_Measurements_full.csv', index=False)


In [6]:
final_columns = [
    col for col in combined_df.columns 
    if col == 'ResponseId' or (
        (col.startswith('Post_') or col.startswith('Base_')) 
        and not col.endswith('_explanation')
    )
]

df_final = combined_df[final_columns]
df_final.to_stata('../input/SwissSurvey_Analysis_GPT_Measurements_Scores_Only.dta', write_index=False, version=118)

## AI_ness (pangram)

In [ ]:
import pandas as pd

df = pd.read_csv('dataset.csv')

cols_to_keep = [
    "ResponseId", 
    "Post", 
    "ClimateLong", 
    "Post-meaningfulness", 
    "ClimateLong-meaningfulness"
]

df_filtered = df[cols_to_keep]
df_filtered = df_filtered.dropna(subset=['Post-meaningfulness', 'ClimateLong-meaningfulness'], how='all')
df_filtered = df_filtered.reset_index(drop=True)
df_filtered[["ResponseId", "ClimateLong", "Post"]].to_csv('SwissSurvey_Id_Base_Post.csv', index=False)
# df_filtered = df_filtered[~((df_filtered['Post-meaningfulness'] == 0) & 
#                              (df_filtered['ClimateLong-meaningfulness'] == 0))]

# df_filtered = df_filtered.reset_index(drop=True)


In [ ]:
import pandas as pd
import numpy as np

df3 = pd.read_csv('SwissSurvey_Id_Base_Post_Pangram.csv')
df3.columns = [col.replace('Climate_', 'Base_') if col.startswith('Climate_') else col for col in df3.columns]
df3 = df3.drop(columns=['ClimateLong', 'Post'])
df3 = df3.rename(columns={
    'Base_headline': 'Base_pangram_headline',
    'Post_headline': 'Post_pangram_headline'
})

# Recode the categories to numeric values
recode_mapping = {
    'Fully AI Generated': 1,
    'AI Detected': 2,
    'AI Assisted': 3,
    'Fully Human Written': 4
}
df['Post_pangram_headline'] = df['Post_pangram_headline'].replace(recode_mapping)
# Using .replace() will map the strings to numbers and safely ignore existing NaNs
df3['Base_pangram_headline'] = df3['Base_pangram_headline'].replace(recode_mapping)
df3['Post_pangram_headline'] = df3['Post_pangram_headline'].replace(recode_mapping)

cols_to_check = [col for col in df3.columns if col != 'ResponseId']

for col in cols_to_check:
    df3[col] = pd.to_numeric(df3[col], errors='coerce')

print("Data types after cleaning:")
print(df3.dtypes)


print("\nMissing values count per column:")
print(df3[cols_to_check].isna().sum())

Data types after cleaning:
ResponseId                    object
Base_pangram_headline          int64
Base_fraction_ai             float64
Base_fraction_ai_assisted    float64
Base_fraction_human          float64
Post_pangram_headline        float64
Post_fraction_ai             float64
Post_fraction_ai_assisted    float64
Post_fraction_human          float64
dtype: object

Missing values count per column:
Base_pangram_headline          0
Base_fraction_ai               0
Base_fraction_ai_assisted      0
Base_fraction_human            0
Post_pangram_headline        159
Post_fraction_ai             159
Post_fraction_ai_assisted    159
Post_fraction_human          159
dtype: int64


C:\Users\hp\AppData\Local\Temp\ipykernel_29628\3908364282.py:21: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df3['Base_pangram_headline'] = df3['Base_pangram_headline'].replace(recode_mapping)
C:\Users\hp\AppData\Local\Temp\ipykernel_29628\3908364282.py:22: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df3['Post_pangram_headline'] = df3['Post_pangram_headline'].replace(recode_mapping)


In [ ]:
df3.to_stata('../input/SwissSurvey_Analysis_Pangram_Recoded.dta', write_index=False, version=118)